# 07 - 模型评估与交叉验证

## 学习目标
- 掌握各种评估指标的含义
- 理解交叉验证的原理
- 学会使用 learning curve 诊断模型

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import (
    cross_val_score, cross_validate, learning_curve, validation_curve,
    StratifiedKFold, KFold
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

## 1. 分类评估指标

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X[:400], y[:400])
y_pred = model.predict(X[400:])

print(f'Accuracy:  {accuracy_score(y[400:], y_pred):.4f}')
print(f'Precision: {precision_score(y[400:], y_pred):.4f}')
print(f'Recall:    {recall_score(y[400:], y_pred):.4f}')
print(f'F1 Score:  {f1_score(y[400:], y_pred):.4f}')

## 2. 交叉验证

In [ ]:
# K-Fold 交叉验证
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
print(f'5-Fold CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')
print(f'各折得分: {scores.round(4)}')

# 多指标交叉验证
scoring = ['accuracy', 'precision', 'recall', 'f1']
cv_results = cross_validate(model, X, y, cv=kf, scoring=scoring)

print('\n多指标结果:')
for metric in scoring:
    mean = cv_results[f'test_{metric}'].mean()
    std = cv_results[f'test_{metric}'].std()
    print(f'  {metric:10s}: {mean:.4f} ± {std:.4f}')

## 3. 学习曲线

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    model, X, y, cv=kf, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1)
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1)
plt.plot(train_sizes, train_mean, 'o-', label='训练集')
plt.plot(train_sizes, val_mean, 's-', label='验证集')
plt.xlabel('训练样本数')
plt.ylabel('准确率')
plt.title('学习曲线')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. 验证曲线

In [ ]:
param_range = np.logspace(-3, 3, 20)
train_scores, val_scores = validation_curve(
    LogisticRegression(max_iter=1000), X, y,
    param_name='C', param_range=param_range,
    cv=kf, n_jobs=-1
)

plt.figure(figsize=(10, 6))
plt.semilogx(param_range, train_scores.mean(axis=1), label='训练集')
plt.semilogx(param_range, val_scores.mean(axis=1), label='验证集')
plt.xlabel('C (正则化参数)')
plt.ylabel('准确率')
plt.title('验证曲线')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 小结
- Accuracy 不适合不平衡数据
- Precision/Recall/F1 根据业务选择
- 交叉验证提供更可靠的评估
- 学习曲线诊断过拟合/欠拟合

---
第一阶段完成！进入进阶: [02_sklearn_advanced/](../02_sklearn_advanced/)